# Gold news and monthly sentiment

**Research question.** Does gold-related news provide information beyond historical market variables when forecasting gold returns and allocating a five-commodity portfolio?

This section prepares one monthly news indicator for the common portfolio experiment. The forecasts, market regimes and portfolio backtest will use the same calendar and must be estimated **without looking ahead**. A useful news signal is a hypothesis, not a presumed result.

## 1. Data choice

The initial source audit established the following:

| Candidate | Audit result | Decision |
|---|---|---|
| Polibert oil news | 11,068 records (2019–2026); 498 duplicate date–headline pairs and inconsistent scoring sources | Exclude from the main experiment |
| The Guardian | 6,851 retained articles across 2000–2025; broad coverage but noisy headlines and no price-direction labels | Exclude from the main experiment |
| [SaguaroCapital gold news](https://huggingface.co/datasets/SaguaroCapital/sentiment-analysis-in-commodity-market-gold) | Human-annotated gold-price direction; historical overlap with World Bank–FRED | **Selected for further evaluation** |

The selected data are associated with [Sinha and Khandait (2020)](https://arxiv.org/abs/2009.04202). Their annotations describe price direction **mentioned in headlines**, often past movements, and are not necessarily forecasts. We study **April 2006–December 2018** to align with the market data. No private API key or local news file is required.

The Hugging Face `train`/`test` splits concern headline classification, **not** the chronological portfolio evaluation. We combine them before monthly aggregation. The audit counts above describe the previous exploratory downloads; they are not recomputed here.

In [ ]:
%pip install -q pandas matplotlib datasets

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset

DATASET_ID = "SaguaroCapital/sentiment-analysis-in-commodity-market-gold"
DATASET_REVISION = "4423810af0fddf2c91a3c9634bbb6d4afdf9d5a5"  # stable snapshot
START, END = pd.Timestamp("2006-04-01"), pd.Timestamp("2018-12-31")
MONTHS = pd.period_range("2006-04", "2018-12", freq="M")
LABELS = ["positive", "negative", "neutral", "none"]
SEED = 42

## 2. Load and audit the headlines

We check the **actual dates**, label values, missing fields and duplicate annotations before selecting the study period. Dates are supplied as day-month-year strings. Malformed dates are reported rather than silently interpreted as valid observations.

In [ ]:
splits = load_dataset(DATASET_ID, revision=DATASET_REVISION)
raw = pd.concat([part.to_pandas() for part in splits.values()], ignore_index=True)
required = {"Dates", "News", "Price Sentiment", "Past Information", "Future Information"}
assert required.issubset(raw.columns), f"Missing columns: {required - set(raw.columns)}"

raw["date"] = pd.to_datetime(
    raw["Dates"].astype("string").str.strip(), format="%d-%m-%Y", errors="coerce"
)
raw["headline"] = raw["News"].astype("string").str.strip()
raw["label"] = raw["Price Sentiment"].astype("string").str.strip().str.lower()

print(f"Downloaded: {len(raw):,} rows")
print(f"Observed dates: {raw['date'].min().date()} to {raw['date'].max().date()}")
print(f"Unparseable dates: {raw['date'].isna().sum()}")
print(f"Missing headlines: {(raw['headline'].isna() | raw['headline'].eq('')).sum()}")
print(f"Missing labels: {raw['label'].isna().sum()}")
if raw["date"].isna().any():
    print("Examples of unparseable dates:", raw.loc[raw["date"].isna(), "Dates"].head(5).tolist())
print("Labels:")
print(raw["label"].value_counts(dropna=False).to_string())

In [ ]:
# The known malformed 2020/2021 dates lie outside our 2006–2018 analysis.
news = raw.loc[raw["date"].between(START, END)].copy()
news = news.dropna(subset=["headline", "label"])
news = news.loc[news["headline"].ne("")].copy()
if news.empty:
    raise ValueError("No valid gold headlines in the selected study period")
assert news["label"].isin(LABELS).all(), "Unexpected label in study period"

# Normalize typography only to identify exact same-day headline duplicates.
news["headline_key"] = (
    news["headline"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
)
keys = ["date", "headline_key"]
conflict = news.groupby(keys)["label"].transform("nunique") > 1
n_conflict = int(conflict.sum())
news = news.loc[~conflict].drop_duplicates(keys).copy()
news["month"] = news["date"].dt.to_period("M")

print(f"Retained in study period: {len(news):,} unique headlines")
print(f"Conflicting duplicate rows excluded: {n_conflict}")
print(f"Observed months: {news['month'].nunique()} / {len(MONTHS)}")
print(f"Past-information share: {news['Past Information'].eq(1).mean():.1%}")
print(f"Future-information share: {news['Future Information'].eq(1).mean():.1%}")

# Small reproducible spot check; human annotation does not guarantee perfect labels.
display(news[["headline", "label", "Past Information", "Future Information"]]
        .sample(min(8, len(news)), random_state=SEED).reset_index(drop=True))

## 3. Monthly directional sentiment

For each headline, assign +1 to `positive`, −1 to `negative` and 0 to `neutral` or `none`. The last two labels have **different meanings** in the source; they contribute zero to this particular index but remain separate in the label counts.

For month \(t\), define the **directional balance**

\[
S_t=\frac{N_t^{+}-N_t^{-}}{N_t^{\mathrm{all}}}, \qquad -1\le S_t\le 1.
\]

Here \(N_t^{\mathrm{all}}\) includes all four classes. This is our aggregation choice, **not a formula supplied by the dataset authors**. It measures how common positive versus negative price-direction statements are; it does not measure an expected percentage return.

In [ ]:
label_counts = pd.crosstab(news["month"], news["label"])
label_counts = label_counts.reindex(index=MONTHS, columns=LABELS, fill_value=0)
n_all = label_counts.sum(axis=1)

monthly_news = pd.DataFrame(index=MONTHS)
monthly_news["gold_news_sentiment"] = (
    (label_counts["positive"] - label_counts["negative"])
    .div(n_all.replace(0, float("nan")))
)
monthly_news["gold_news_count"] = n_all
monthly_news["gold_news_directional_share"] = (
    (label_counts["positive"] + label_counts["negative"])
    .div(n_all.replace(0, float("nan")))
)

assert monthly_news["gold_news_sentiment"].dropna().between(-1, 1).all()
print(f"Calendar months: {len(monthly_news)}")
print(f"Months without headlines: {(n_all == 0).sum()}")
print(f"Median headlines/month: {n_all.median():.0f}")
display(monthly_news.describe().round(3))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 5.4), sharex=True)
monthly_news["gold_news_count"].plot(ax=axes[0])
axes[0].set(title="Gold news coverage, 2006–2018", ylabel="Headlines / month", xlabel="")
monthly_news["gold_news_sentiment"].plot(ax=axes[1])
axes[1].axhline(0, linewidth=0.8, color="gray")
axes[1].set(title="Monthly directional balance", ylabel="Sentiment index", xlabel="Month")
plt.tight_layout()
plt.show()

## 4. Timing and integration

A headline published in month \(t\) can reflect the price movement **during that same month**. To match the conservative timing rule in the portfolio analysis, month-\(t\) news is used only for an allocation in month \(t+2\).

Therefore, when rows are indexed by the **portfolio outcome month** \(u\), the feature is \(S_{u-2}\). We use a calendar-indexed shift, not a row shift on an irregular subset. This safeguards information timing; it does not guarantee that historical headlines predict returns.

In [ ]:
# Month-start dates match the commodity and FRED monthly indexes.
monthly_news.index = MONTHS.to_timestamp(how="start")
monthly_news.index.name = "date"
monthly_news = monthly_news.asfreq("MS")

# The feature used for return month u is sentiment observed in u-2.
news_features = monthly_news[["gold_news_sentiment"]].shift(2).rename(
    columns={"gold_news_sentiment": "gold_sentiment_lag2"}
)
assert news_features.index.is_monotonic_increasing
assert news_features.index.freqstr == "MS"
assert news_features["gold_sentiment_lag2"].iloc[:2].isna().all()

print("Feature dates:", news_features.index.min().date(),
      "to", news_features.index.max().date())
display(news_features.head(5))

## 5. Interpretation and next step

This pipeline produces **one additional predictor**, `news_features['gold_sentiment_lag2']`, indexed by the portfolio return month. It can be joined to the common World Bank–FRED feature table inside the final notebook. The first two values are intentionally missing because the signal has a two-month lag.

The analysis does **not** yet establish predictive value. Three concerns remain: the annotations mostly describe past/current movements; the corpus may contain ambiguous or imperfect labels; and historical news coverage can change across publishers and years. The appropriate next test compares **otherwise identical gold-return models with and without this one feature**, then evaluates the impact on portfolio performance using one shared backtest and chronological validation.

*Sources: [SaguaroCapital dataset](https://huggingface.co/datasets/SaguaroCapital/sentiment-analysis-in-commodity-market-gold); Sinha & Khandait (2020), [Impact of News on the Commodity Market: Dataset and Results](https://arxiv.org/abs/2009.04202).*